In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
%reload_ext autoreload

In [58]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [ ]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [60]:
from dotenv import load_dotenv
from tree_of_thought import TreeOfThought
import openai
import os, torch, numpy as np
from utils import call_llm
from plot_tree import plot_tree
import json
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4o-mini"

In [ ]:
import pandas as pd
from tqdm import tqdm 
import os, json
from stereotype_definitions import stereotype_definition_short_binary as stereotype_definition
from tree_of_thought import TreeOfThought
from cases import stereotypes_case, CaseConfig, ThoughtOutput


token_regime_name = 'medium'
max_branching_factor = 2
max_depth=3

token_regimes = {
    'low': {
        'generation': 250,
        'evaluation': 5
    },
    'medium': {
        'generation': 500,
        'evaluation': 10
    },
    'high': {
        'generation': 750,
        'evaluation': 20
    }
}

tot_stereotype = TreeOfThought(
    case=stereotypes_case,
    client=client,
    model=model,
    max_branching_factor=max_branching_factor,
    max_depth=max_depth,
    task_definition=stereotype_definition,
    max_tokens_dict=token_regimes[token_regime_name],
    examples_df=sample_examples_mgsd,
    n_shots=1, 
    reasoning_budget={
        "effort": token_regime_name,
        "summary": None,
    }
)
import pandas as pd
from tqdm import tqdm
import os, json
import time
from openai import RateLimitError


output_dir = f"results/tot/judge_inline/{token_regime_name}"
csv_path = f"{output_dir}/results_stereotype_{max_depth}_{max_branching_factor}.csv"
json_path = f"{output_dir}/tree_reasoning_{max_depth}_{max_branching_factor}.json"
os.makedirs(output_dir, exist_ok=True)


start_index = 0
if os.path.exists(csv_path):
    df_out_existing = pd.read_csv(csv_path)
    done_ids = set(df_out_existing['sample_id'])
    rows = df_out_existing.to_dict(orient="records")
    detailed_outputs = json.load(open(json_path)) if os.path.exists(json_path) else []
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    rows = []
    detailed_outputs = []
    done_ids = set()

try:
    for idx, row in tqdm(sample_mgsd.iterrows(), total=len(sample_mgsd)):
        if idx in done_ids:
            continue

        text = row["text_no_marker"]
        true_label = row["label"].strip()

        tot_stereotype.total_tokens = 0
        tot_stereotype.total_prompt_tokens = 0
        tot_stereotype.total_completion_tokens = 0
        tot_stereotype.total_latency = 0.0
        tot_stereotype.total_calls = 0
        tot_stereotype.tie_events = 0
        tot_stereotype.tie_pairs = 0
        tot_stereotype.max_tie_group = 0

        try:
            solution = tot_stereotype.solve(text)
        except RateLimitError as e:
            print(f"\n Rate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\n Error at sample {idx}: {e}. Skipping.")
            continue

        path_vote = tot_stereotype._get_majority_vote_from_path(solution)
        tree_vote = tot_stereotype._get_majority_vote_from_tree()
        leaf_vote = tot_stereotype._get_majority_vote_from_leafs()

        w_path_vote = tot_stereotype._get_majority_vote_from_path(solution, weighted=True)
        w_tree_vote = tot_stereotype._get_majority_vote_from_tree(weighted=True)
        w_leaf_vote = tot_stereotype._get_majority_vote_from_leafs(weighted=True)

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_path": tot_stereotype.map_label(path_vote),
            "pred_tree": tot_stereotype.map_label(tree_vote),
            "pred_leaf": tot_stereotype.map_label(leaf_vote),
            "w_pred_path": tot_stereotype.map_label(w_path_vote),
            "w_pred_tree": tot_stereotype.map_label(w_tree_vote),
            "w_pred_leaf": tot_stereotype.map_label(w_leaf_vote),
            "llm_calls": tot_stereotype.total_calls,
            "tokens_prompt": tot_stereotype.total_prompt_tokens,
            "tokens_completion": tot_stereotype.total_completion_tokens,
            "tokens_total": tot_stereotype.total_tokens,
            "latency_total": round(tot_stereotype.total_latency, 2),
            "tie_events": tot_stereotype.tie_events,
            "tie_pairs": tot_stereotype.tie_pairs,
            "max_tie_group": tot_stereotype.max_tie_group,
            "token_regime": token_regime_name,
            "generation_tokens": tot_stereotype.max_tokens_dict["generation"],
            "evaluation_tokens": tot_stereotype.max_tokens_dict["evaluation"],
        }

        tree_output = {
            "sample_id": idx,
            "input_text": text,
            "true_label": true_label,
            "predicted_path_label": tot_stereotype.map_label(path_vote),
            "predicted_tree_label": tot_stereotype.map_label(tree_vote),
            "predicted_leaf_label": tot_stereotype.map_label(leaf_vote),
            "tree": tot_stereotype.get_tree_dict(),
            "tie_info": {
                "tie_events": tot_stereotype.tie_events,
                "tie_pairs": tot_stereotype.tie_pairs,
                "max_tie_group": tot_stereotype.max_tie_group,
            },
            "token_usage": {
                "prompt_tokens": tot_stereotype.total_prompt_tokens,
                "completion_tokens": tot_stereotype.total_completion_tokens,
                "total_tokens": tot_stereotype.total_tokens
            },
            "latency": tot_stereotype.total_latency
        }

        rows.append(results)
        detailed_outputs.append(tree_output)

        if (idx + 1) % 20 == 0:
            print(f"\n===== FULL TREE for sample {idx} =====\n")
            tot_stereotype.print_full_tree()
            print("\n======================================\n")

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(detailed_outputs, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(rows)} samples to disk.")


=== Resuming from last index... 154 samples already completed.


 32%|███▏      | 160/500 [02:22<12:43,  2.25s/it]


===== FULL TREE for sample 159 =====

 Thought ID 0
 --  Score: 0.00
 -- Content: There tends to be a lot of koreans in academia because they are smart.
------------------------------------------------------------
     Thought ID 0.1
     --  Score: 0.60
     -- Content: The statement implies that all Koreans are smart, which is a generalization based on their nationality.
    ------------------------------------------------------------
         Thought ID 0.1.1
         --  Score: 0.40
         -- Content: The statement makes a broad claim about the intelligence of all Koreans, which is a generalization based on their nationality.
        ------------------------------------------------------------
             Thought ID 0.1.1.1
             --  Score: 0.80
             -- Content: The statement generalizes about the intelligence of all Koreans, which is a clear stereotype based on nationality.
             -- Label: Yes
            --------------------------------------------------

 33%|███▎      | 167/500 [06:51<13:40,  2.46s/it]  


 Rate limit hit at sample 167. Saving progress.
✅ Saved 167 samples to disk.


In [80]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

label_studied = "w_pred_leaf"
df = pd.read_csv(f"results/tot/judge_inline/{token_regime_name}/results_stereotype_{max_depth}_{max_branching_factor}.csv")
y_true = df["true_label"].str.lower()
y_pred = df[label_studied].str.lower() 

print(f"=== Classification Report ({label_studied} Vote) ===\n")
print(classification_report(y_true, y_pred))

labels = sorted(y_true.unique())
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print("=== Confusion Matrix ===")
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

=== Classification Report (w_pred_leaf Vote) ===

              precision    recall  f1-score   support

  stereotype       0.67      0.66      0.67        86
   unrelated       0.65      0.65      0.65        81

    accuracy                           0.66       167
   macro avg       0.66      0.66      0.66       167
weighted avg       0.66      0.66      0.66       167

=== Confusion Matrix ===
            stereotype  unrelated
stereotype          57         29
unrelated           28         53


In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from manipulation_definitions import manipulation_definition, manipulation_definition_short
from case_studies import manipulation
from tree_of_thought import TreeOfThought


token_regime_name = 'medium'
max_branching_factor=2
max_depth=3

token_regimes = {
    'low': {
        'generation': 250,
        'evaluation': 35
    },
    'medium': {
        'generation': 500,
        'evaluation': 50
    },
    'high': {
        'generation': 750,
        'evaluation': 75
    }
}


tot_manip = TreeOfThought(
    case=manipulation,
    client=client,
    model=model,
    max_branching_factor=max_branching_factor,
    max_depth=max_depth,
    task_definition= manipulation_definition_short,
    max_tokens_dict=token_regimes[token_regime_name]
)

rows = []

for idx, row in tqdm(sample_mentalmanip.iterrows(), total=len(sample_mentalmanip)):
    text = row["dialogue"]
    true_label = row["manipulative"]

    tot_manip.total_tokens            = 0
    tot_manip.total_prompt_tokens     = 0
    tot_manip.total_completion_tokens = 0
    tot_manip.total_latency           = 0.0
    tot_manip.total_calls             = 0
    tot_manip.tie_events              = 0
    tot_manip.tie_pairs               = 0
    tot_manip.max_tie_group           = 0

    solution = tot_manip.solve(text)

    path_vote  = tot_manip._get_majority_vote_from_path(solution)
    tree_vote  = tot_manip._get_majority_vote_from_tree()
    leaf_vote  = tot_manip._get_majority_vote_from_leafs()

    w_path_vote  = tot_manip._get_majority_vote_from_path(solution, weighted=True)
    w_tree_vote  = tot_manip._get_majority_vote_from_tree(weighted=True)
    w_leaf_vote  = tot_manip._get_majority_vote_from_leafs(weighted=True)

    results = {
        "sample_id":      idx,
        "text":           text,
        "true_label":     true_label,

        "pred_path": tot_manip.map_label(path_vote),
        "pred_tree": tot_manip.map_label(tree_vote),
        "pred_leaf": tot_manip.map_label(leaf_vote),

        "w_pred_path": tot_manip.map_label(w_path_vote),
        "w_pred_tree": tot_manip.map_label(w_tree_vote),
        "w_pred_leaf": tot_manip.map_label(w_leaf_vote),

        "llm_calls":      tot_manip.total_calls,
        "tokens_prompt":  tot_manip.total_prompt_tokens,
        "tokens_completion": tot_manip.total_completion_tokens,
        "tokens_total":   tot_manip.total_tokens,
        "latency_total":  round(tot_manip.total_latency, 2),

        "tie_events":     tot_manip.tie_events,
        "tie_pairs":      tot_manip.tie_pairs,
        "max_tie_group":  tot_manip.max_tie_group,

        "token_regime":      token_regime_name,
        "generation_tokens": tot_manip.max_tokens_dict["generation"],
        "evaluation_tokens": tot_manip.max_tokens_dict["evaluation"],
    }

    rows.append(results)

    if (idx+1) % 20 == 0:
        print(f"\n===== FULL TREE for sample {idx} =====\n")
        tot_manip.print_full_tree()
        print("\n======================================\n")

df_out = pd.DataFrame(rows)
df_out.to_csv(f"results/{token_regime_name}/results_mentalmanip_{max_depth}_{max_branching_factor}.csv", index=False)
print("=== Saved", len(df_out), f"rows to results_mentalmanip_{max_depth}_{max_branching_factor}.csv")


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

result_studied = "w_pred_leaf"

df = pd.read_csv(f"results/{token_regime_name}/results_mentalmanip_{max_depth}_{max_branching_factor}.csv")
y_true = df["true_label"]
y_pred = df[result_studied]

print(f"=== Classification Report ({result_studied} Vote) ===\n")
print(classification_report(y_true, y_pred))

labels = sorted(y_true.unique())
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print("=== Confusion Matrix ===")
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

result_studied = "w_pred_tree"

df = pd.read_csv(f"results/{token_regime_name}/results_mentalmanip_{max_depth}_{max_branching_factor}.csv")
y_true = df["true_label"]
y_pred = df[result_studied]

print(f"=== Classification Report ({result_studied} Vote) ===\n")
print(classification_report(y_true, y_pred))

labels = sorted(y_true.unique())
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print("=== Confusion Matrix ===")
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

result_studied = "w_pred_path"

df = pd.read_csv(f"results/{token_regime_name}/results_mentalmanip_{max_depth}_{max_branching_factor}.csv")
y_true = df["true_label"]
y_pred = df[result_studied]

print(f"=== Classification Report ({result_studied} Vote) ===\n")
print(classification_report(y_true, y_pred))

labels = sorted(y_true.unique())
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print("=== Confusion Matrix ===")
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

result_studied = "pred_tree"

df = pd.read_csv(f"results/{token_regime_name}/results_mentalmanip_{max_depth}_{max_branching_factor}.csv")
y_true = df["true_label"]
y_pred = df[result_studied]

print(f"=== Classification Report ({result_studied} Vote) ===\n")
print(classification_report(y_true, y_pred))

labels = sorted(y_true.unique())
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print("=== Confusion Matrix ===")
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from stereotype_def import stereotype_definition, stereotype_definition_short
from manipulation_definitions import manipulation_definition, manipulation_definition_short
from case_studies import stereotypes, manipulation
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix


case_name = "stereotype" # "stereotype" "manipulation"

if case_name.lower() == "manipulation":
    case = manipulation
    task_definition = manipulation_definition_short
    data = sample_mentalmanip
    text_col = "dialogue"
    label_col = "manipulative"
    output_file = "results/low/results_mentalmanip_zero_shot_prompt_long.csv"

elif case_name.lower() == "stereotype":
    case = stereotypes
    task_definition = stereotype_definition
    data = sample_mgsd
    text_col = "text_no_marker"
    label_col = "label"
    output_file = "results/low/results_stereotype_zero_shot_prompt_long.csv"

else:
    raise ValueError(f"Unknown case name: {case_name}")


zero_shot_classifier = ZeroShot(
    case=case,
    client=client,
    model=model,
    max_tokens=100,
    task_definition=task_definition
)


rows = []


for idx, row in tqdm(data.iterrows(), total=len(data)):
    text = row[text_col]
    true_label = row[label_col]
    
    if isinstance(true_label, str):
        true_label = true_label.strip()

    predicted_label = zero_shot_classifier.classify(text)
    mapped_label = case["label_map"].get(predicted_label.strip(), list(case["label_map"].values())[-1])

    results = {
        "sample_id": idx,
        "text": text,
        "true_label": true_label,
        "pred_label": mapped_label,
        "max_tokens": zero_shot_classifier.max_tokens,
        "tokens_used": zero_shot_classifier.total_tokens,
        "prompt_tokens": zero_shot_classifier.total_prompt_tokens,
        "completion_tokens": zero_shot_classifier.total_completion_tokens,
        "latency": zero_shot_classifier.total_latency,
    }

    rows.append(results)

df_out = pd.DataFrame(rows)
df_out.to_csv(output_file, index=False)
print(f"=== Saved {len(df_out)} rows to {output_file}")

if case_name == "manipulation":
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

elif case_name == "stereotype":
    y_true = df_out["true_label"]
    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()


print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred))


print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true == y_pred).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")


## Stereotypes results

In [ ]:
import pandas as pd
import os
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Parameters
results_dir = "results"
evaluation_methods = ["w_pred_path", "w_pred_tree", "w_pred_leaf"]
max_depth = 3
max_branching_factor = 2

# Load all Tree of Thoughts result files
records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_stereotype_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

# Combine all data
tot_results = pd.concat(records, ignore_index=True)

# Evaluation loop
for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"].str.lower()
        y_pred = subset[method].str.lower()

        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"].str.lower()
            y_pred = subset[method].str.lower()
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices by Regime and Vote Method", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"].str.lower() == tot_results[method].str.lower() for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}_acc": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))

In [ ]:
import pandas as pd
import os
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Parameters
results_dir = "results"
evaluation_methods = ["pred_path", "pred_tree", "pred_leaf"]
max_depth = 3
max_branching_factor = 2

# Load all Tree of Thoughts result files
records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_stereotype_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

# Combine all data
tot_results = pd.concat(records, ignore_index=True)

# Evaluation loop
for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"].str.lower()
        y_pred = subset[method].str.lower()

        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"].str.lower()
            y_pred = subset[method].str.lower()
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices by Regime and Vote Method", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Accuracy summary table
summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"].str.lower() == tot_results[method].str.lower() for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}_acc": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))


## Results Manipulation

In [ ]:
import pandas as pd
import os
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Parameters
results_dir = "results"
evaluation_methods = ["w_pred_path", "w_pred_tree", "w_pred_leaf"]
max_depth = 3
max_branching_factor = 2

# Load all Tree of Thoughts result files
records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_manipulation_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

# Combine all data
tot_results = pd.concat(records, ignore_index=True)

# Ensure numeric columns
tot_results["true_label"] = tot_results["true_label"].astype(int)
for method in evaluation_methods:
    tot_results[method] = tot_results[method].astype(int)

# Evaluation loop
for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"]
        y_pred = subset[method]
        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = [0, 1]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"]
            y_pred = subset[method]
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices (Manipulation)", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Accuracy summary table
summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"] == tot_results[method] for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}_acc": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))



In [ ]:
import pandas as pd
import os
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Parameters
results_dir = "results"
evaluation_methods = ["pred_path", "pred_tree", "pred_leaf"]
max_depth = 3
max_branching_factor = 2

# Load all Tree of Thoughts result files
records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_manipulation_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

# Combine all data
tot_results = pd.concat(records, ignore_index=True)

# Ensure numeric columns
tot_results["true_label"] = tot_results["true_label"].astype(int)
for method in evaluation_methods:
    tot_results[method] = tot_results[method].astype(int)

# Evaluation loop
for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"]
        y_pred = subset[method]
        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = [0, 1]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"]
            y_pred = subset[method]
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices (Manipulation)", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Accuracy summary table
summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"] == tot_results[method] for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))
